<a href="https://colab.research.google.com/github/cauarichard/deepfakes/blob/main/id0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==============================
# 0) MONTAR O GOOGLE DRIVE
# ==============================
from google.colab import drive
drive.mount('/content/drive')

# Depois de montar, rode o resto desta célula normalmente
import os
import shutil
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from torchvision import transforms
from torch.utils.data import Dataset, DataLoader, random_split
from PIL import Image

from sklearn.metrics import roc_auc_score, f1_score, roc_curve

from transformers import AutoImageProcessor, AutoModel


# ==============================
# 1) CAMINHOS E CONFIGURAÇÕES
# ==============================
BASE_DIR = "/content/drive/MyDrive/dataset/real_fake/videos_celeb"
FACES_REAIS_ROOT = os.path.join(BASE_DIR, "faces_reais")
FACES_FAKES_ROOT = os.path.join(BASE_DIR, "faces_fakes")

MODELS_ROOT = "/content/drive/MyDrive/dataset/modelos_dinov2"
os.makedirs(MODELS_ROOT, exist_ok=True)

DINOV2_MODEL_NAME = "facebook/dinov2-base"

BATCH_SIZE = 64
NUM_EPOCHS = 50
LR = 1e-3

PATIENCE = 7
MIN_DELTA = 1e-4

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


# ==============================
# 2) DINOv2 (CONGELADO)
# ==============================
print("Carregando DINOv2...")
image_processor = AutoImageProcessor.from_pretrained(DINOV2_MODEL_NAME)
dinov2_backbone = AutoModel.from_pretrained(DINOV2_MODEL_NAME)
dinov2_backbone.to(device)
dinov2_backbone.eval()

for p in dinov2_backbone.parameters():
    p.requires_grad = False

with torch.no_grad():
    from PIL import Image as PILImage
    dummy_image = PILImage.new("RGB", (518, 518), color=(0, 0, 0))
    dummy_inputs = image_processor(images=dummy_image, return_tensors="pt")
    dummy_pixel_values = dummy_inputs["pixel_values"].to(device)

    out = dinov2_backbone(pixel_values=dummy_pixel_values)
    if hasattr(out, "pooler_output") and out.pooler_output is not None:
        EMBED_DIM = out.pooler_output.shape[-1]
        USE_POOLER = True
    else:
        EMBED_DIM = out.last_hidden_state[:, 0].shape[-1]
        USE_POOLER = False

print("Dimensão do embedding DINOv2:", EMBED_DIM)


def dinov2_embeddings(batch_pixels: torch.Tensor) -> torch.Tensor:
    with torch.no_grad():
        outputs = dinov2_backbone(pixel_values=batch_pixels.to(device))
        if USE_POOLER and hasattr(outputs, "pooler_output") and outputs.pooler_output is not None:
            feats = outputs.pooler_output
        else:
            feats = outputs.last_hidden_state[:, 0]
    return feats


# ==============================
# 3) DATASET NOVO
# ==============================
class FacesDinoDatasetV2(Dataset):
    def __init__(self, folder_real, folder_fake, augment=False):
        self.paths = []
        self.labels = []
        self.augment = augment

        if os.path.isdir(folder_real):
            for f in os.listdir(folder_real):
                p = os.path.join(folder_real, f)
                if os.path.isfile(p):
                    self.paths.append(p)
                    self.labels.append(1)

        if os.path.isdir(folder_fake):
            for f in os.listdir(folder_fake):
                p = os.path.join(folder_fake, f)
                if os.path.isfile(p):
                    self.paths.append(p)
                    self.labels.append(0)

        if augment:
            self.transform = transforms.Compose([
                transforms.RandomHorizontalFlip(),
                transforms.RandomRotation(12),
            ])
        else:
            self.transform = None

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]
        label = self.labels[idx]
        img = PILImage.open(path).convert("RGB")

        if self.transform is not None:
            img = self.transform(img)

        inputs = image_processor(images=img, return_tensors="pt")
        pixel_values = inputs["pixel_values"].squeeze(0)

        return pixel_values, label


# ==============================
# 4) MLP CLASSIFIER
# ==============================
class DinoMLP(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.fc1 = nn.Linear(embed_dim, 512)
        self.fc2 = nn.Linear(512, 128)
        self.fc3 = nn.Linear(128, 1)
        self.drop1 = nn.Dropout(0.5)
        self.drop2 = nn.Dropout(0.5)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.drop1(x)
        x = F.relu(self.fc2(x))
        x = self.drop2(x)
        x = torch.sigmoid(self.fc3(x))
        return x


# ==============================
# 5) MÉTRICAS
# ==============================
def calcular_metricas(y_true_tensor, y_prob_tensor):
    y_true = y_true_tensor.detach().cpu().numpy()
    y_prob = y_prob_tensor.detach().cpu().numpy()

    try:
        auc = roc_auc_score(y_true, y_prob)
    except ValueError:
        auc = float("nan")

    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    fnr = 1 - tpr

    abs_diffs = abs(fpr - fnr)
    idx_eer = abs_diffs.argmin()
    eer = (fpr[idx_eer] + fnr[idx_eer]) / 2.0
    hter = eer

    y_pred_bin = (y_prob >= 0.5).astype("int32")
    f1 = f1_score(y_true, y_pred_bin, zero_division=0)

    return auc, eer, hter, f1


# ==============================
# 6) LISTAR IDs
# ==============================
def listar_ids_faces():
    ids_real = []
    ids_fake = []

    if os.path.isdir(FACES_REAIS_ROOT):
        ids_real = [d for d in os.listdir(FACES_REAIS_ROOT)
                    if os.path.isdir(os.path.join(FACES_REAIS_ROOT, d))]
    if os.path.isdir(FACES_FAKES_ROOT):
        ids_fake = [d for d in os.listdir(FACES_FAKES_ROOT)
                    if os.path.isdir(os.path.join(FACES_FAKES_ROOT, d))]

    ids_real_base = {d.split("_")[0] for d in ids_real}
    ids_fake_base = {d.split("_")[0] for d in ids_fake}

    ids_todos = sorted(ids_real_base | ids_fake_base)
    print("IDs disponíveis:")
    for i in ids_todos:
        print("-", i)
    return ids_todos


# ==============================
# 7) TREINO POR ID
# ==============================
def treinar_id_com_dinov2():
    ids_todos = listar_ids_faces()
    if not ids_todos:
        print("Nenhum ID encontrado.")
        return

    celeb_id = input("\nDigite o ID para treinar (ex: id0): ").strip()
    if celeb_id not in ids_todos:
        print("ID não encontrado.")
        return

    folder_real = os.path.join(FACES_REAIS_ROOT, f"{celeb_id}_faces_real")
    folder_fake = os.path.join(FACES_FAKES_ROOT, f"{celeb_id}_faces_fake")

    print("Pasta real:", folder_real)
    print("Pasta fake:", folder_fake)

    full_dataset = FacesDinoDatasetV2(folder_real, folder_fake, augment=True)

    if len(full_dataset) < 50:
        print(f"Poucas imagens ({len(full_dataset)}) para {celeb_id}.")
        return

    total = len(full_dataset)
    train_size = int(0.7 * total)
    val_size = int(0.15 * total)
    test_size = total - train_size - val_size

    train_dataset, val_dataset, test_dataset = random_split(
        full_dataset, [train_size, val_size, test_size]
    )

    val_dataset.dataset.augment = False
    test_dataset.dataset.augment = False
    val_dataset.dataset.transform = None
    test_dataset.dataset.transform = None

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

    model = DinoMLP(EMBED_DIM).to(device)
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=LR)

    best_val_auc = -1.0
    best_model_path = os.path.join("/content", f"dinov2_{celeb_id}_best.pth")
    patience_counter = 0

    print("\nIniciando treino...")

    for epoch in range(1, NUM_EPOCHS + 1):
        model.train()
        running_loss = 0.0
        correct = 0
        total_train = 0

        for pixels, labels in train_loader:
            pixels = pixels.to(device)
            labels = labels.to(device).float().unsqueeze(1)

            with torch.no_grad():
                feats = dinov2_embeddings(pixels)

            optimizer.zero_grad()
            outputs = model(feats)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * labels.size(0)
            preds = (outputs > 0.5).float()
            correct += (preds == labels).sum().item()
            total_train += labels.size(0)

        train_loss = running_loss / total_train
        train_acc = correct / total_train * 100.0

        model.eval()
        val_loss = 0.0
        val_correct = 0
        total_val = 0

        all_val_labels = []
        all_val_probs = []

        with torch.no_grad():
            for pixels, labels in val_loader:
                pixels = pixels.to(device)
                labels = labels.to(device).float().unsqueeze(1)

                feats = dinov2_embeddings(pixels)
                outputs = model(feats)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * labels.size(0)
                preds = (outputs > 0.5).float()
                val_correct += (preds == labels).sum().item()
                total_val += labels.size(0)

                all_val_labels.append(labels)
                all_val_probs.append(outputs)

        val_loss = val_loss / total_val
        val_acc = val_correct / total_val * 100.0
        all_val_labels = torch.cat(all_val_labels, dim=0).squeeze(1)
        all_val_probs = torch.cat(all_val_probs, dim=0).squeeze(1)

        val_auc, val_eer, val_hter, val_f1 = calcular_metricas(all_val_labels, all_val_probs)

        print(f"Época {epoch}/{NUM_EPOCHS} | "
              f"Train Loss: {train_loss:.4f}, Acc: {train_acc:.2f}% | "
              f"Val Loss: {val_loss:.4f}, Acc: {val_acc:.2f}% | "
              f"AUC: {val_auc:.4f}, EER: {val_eer:.4f}, "
              f"HTER: {val_hter:.4f}, F1: {val_f1:.4f}")

        improved = val_auc > best_val_auc + MIN_DELTA

        if improved:
            best_val_auc = val_auc
            patience_counter = 0
            torch.save(model.state_dict(), best_model_path)
            print("  -> Melhor AUC até agora, modelo salvo.")
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print("  -> Early stopping ativado.")
                break

    print("\nCarregando melhor modelo salvo para avaliar no TESTE...")
    best_model = DinoMLP(EMBED_DIM).to(device)
    best_model.load_state_dict(torch.load(best_model_path, map_location=device))
    best_model.eval()

    all_test_labels = []
    all_test_probs = []
    test_correct = 0
    total_test = 0

    with torch.no_grad():
        for pixels, labels in test_loader:
            pixels = pixels.to(device)
            labels = labels.to(device).float().unsqueeze(1)

            feats = dinov2_embeddings(pixels)
            outputs = best_model(feats)

            preds = (outputs > 0.5).float()
            test_correct += (preds == labels).sum().item()
            total_test += labels.size(0)

            all_test_labels.append(labels)
            all_test_probs.append(outputs)

    all_test_labels = torch.cat(all_test_labels, dim=0).squeeze(1)
    all_test_probs = torch.cat(all_test_probs, dim=0).squeeze(1)

    test_auc, test_eer, test_hter, test_f1 = calcular_metricas(all_test_labels, all_test_probs)
    test_acc = test_correct / total_test * 100.0

    print(f"\nRESULTADOS FINAIS NO TESTE ({celeb_id}):")
    print(f"Accuracy: {test_acc:.2f}%")
    print(f"AUC:      {test_auc:.4f}")
    print(f"EER:      {test_eer:.4f}")
    print(f"HTER:     {test_hter:.4f}")
    print(f"F1:       {test_f1:.4f}")

    final_path = os.path.join(MODELS_ROOT, f"dinov2_{celeb_id}_best.pth")
    shutil.copy(best_model_path, final_path)
    print("\nMelhor modelo copiado para:", final_path)


# ==============================
# 8) RODAR
# ==============================
treinar_id_com_dinov2()


Mounted at /content/drive
Device: cuda
Carregando DINOv2...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/436 [00:00<?, ?B/s]

The image processor of type `BitImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json:   0%|          | 0.00/548 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Dimensão do embedding DINOv2: 768
IDs disponíveis:
- id0

Digite o ID para treinar (ex: id0): id0
Pasta real: /content/drive/MyDrive/dataset/real_fake/videos_celeb/faces_reais/id0_faces_real
Pasta fake: /content/drive/MyDrive/dataset/real_fake/videos_celeb/faces_fakes/id0_faces_fake

Iniciando treino...
Época 1/50 | Train Loss: 0.2984, Acc: 91.02% | Val Loss: 0.2523, Acc: 91.93% | AUC: 0.8411, EER: 0.2322, HTER: 0.2322, F1: 0.0000
  -> Melhor AUC até agora, modelo salvo.
Época 2/50 | Train Loss: 0.2381, Acc: 91.99% | Val Loss: 0.1990, Acc: 91.93% | AUC: 0.9541, EER: 0.1030, HTER: 0.1030, F1: 0.0000
  -> Melhor AUC até agora, modelo salvo.
Época 3/50 | Train Loss: 0.1664, Acc: 93.10% | Val Loss: 0.1318, Acc: 94.14% | AUC: 0.9791, EER: 0.0729, HTER: 0.0729, F1: 0.4298
  -> Melhor AUC até agora, modelo salvo.
Época 4/50 | Train Loss: 0.1313, Acc: 94.97% | Val Loss: 0.1027, Acc: 95.75% | AUC: 0.9881, EER: 0.0499, HTER: 0.0499, F1: 0.6429
  -> Melhor AUC até agora, modelo salvo.
Época 5/50 

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix

# ATENÇÃO: ajuste esse ID se tiver treinado outro
celeb_id = "id0"

# Recarrega o melhor modelo salvo para esse ID
best_model_path = os.path.join("/content", f"dinov2_{celeb_id}_best.pth")
best_model = DinoMLP(EMBED_DIM).to(device)
best_model.load_state_dict(torch.load(best_model_path, map_location=device))
best_model.eval()

all_test_labels = []
all_test_probs = []
test_correct = 0
total_test = 0

with torch.no_grad():
    for pixels, labels in test_loader:   # usa o mesmo test_loader já criado
        pixels = pixels.to(device)
        labels = labels.to(device).float().unsqueeze(1)

        feats = dinov2_embeddings(pixels)
        outputs = best_model(feats)

        preds = (outputs > 0.5).float()
        test_correct += (preds == labels).sum().item()
        total_test += labels.size(0)

        all_test_labels.append(labels)
        all_test_probs.append(outputs)

all_test_labels = torch.cat(all_test_labels, dim=0).squeeze(1)
all_test_probs = torch.cat(all_test_probs, dim=0).squeeze(1)

# binariza e calcula matriz de confusão
y_true_test = all_test_labels.detach().cpu().numpy().astype(int)
y_prob_test = all_test_probs.detach().cpu().numpy()
y_pred_test = (y_prob_test >= 0.5).astype(int)

tn, fp, fn, tp = confusion_matrix(y_true_test, y_pred_test).ravel()
tpr = tp / (tp + fn + 1e-8)
tnr = tn / (tn + fp + 1e-8)

print("MATRIZ DE CONFUSÃO (TESTE):")
print(f"TN: {tn}, FP: {fp}, FN: {fn}, TP: {tp}")
print(f"Sensitivity (TPR): {tpr:.4f}")
print(f"Specificity (TNR): {tnr:.4f}")
